# RAG進階

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY,
    temperature=0
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL
model_name = 'google/gemma-3-12b'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url,
    temperature=0
)

## 測試模型

In [84]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("機器學習的定義")
]

for chunk in llm.stream(messages):
    print(chunk.content, end='')

## 機器學習 (Machine Learning) 的定義

機器學習是**人工智能（AI）的一個分支，它使計算機系統能夠從數據中學習，而無需進行顯式編程。**  換句話說，我們不是直接告訴電腦如何執行某項任務，而是給它大量的數據，讓它自己找出規律和模式，並根據這些規律做出預測或決策。

更詳細地解釋：

* **傳統的編程 (Traditional Programming):**
    * 我們明確地告訴計算機每一步該怎麼做。
    * 輸入 -> 規則 -> 輸出
* **機器學習 (Machine Learning):**
    * 我們給計算機大量的數據，讓它自己找出規則。
    * 輸入 -> 數據 -> 算法 -> 輸出

**核心概念：**

* **數據 (Data):**  機器學習的基礎，可以是任何形式的信息，例如圖像、文本、數字等。
* **算法 (Algorithm):** 用於分析數據並從中學習的模型或方法。  有很多不同的機器學習算法，每種算法都適合解決不同類型的問題。
* **模型 (Model):** 算法在訓練數據上運行後產生的結果，它代表了從數據中學習到的模式和規律。
* **訓練 (Training):** 使用數據來調整模型的參數，使其能夠做出更準確的預測或決策的過程。
* **預測/推斷 (Prediction/Inference):**  使用訓練好的模型對新的、未見過的數據進行預測或決策。

**機器學習的主要類型：**

* **監督學習 (Supervised Learning):** 使用帶有標籤的數據（即已知輸入和輸出）來訓練模型，例如分類和回歸。
    * **分類 (Classification):**  將數據分到不同的類別中，例如垃圾郵件檢測、圖像識別。
    * **回歸 (Regression):** 預測一個連續值，例如房價預測、股票價格預測。
* **非監督學習 (Unsupervised Learning):** 使用沒有標籤的數據來訓練模型，例如聚類和降維。
    * **聚類 (Clustering):** 將數據分組到不同的簇中，例如客戶分群、異常檢測。
    * **降維 (Dimensionality Reduction):** 減少數據的維度，同時保留重要的信息，例

# 準備資料

## 載入資料

In [ ]:
from langchain_community.document_loaders import DirectoryLoader 
from langchain_community.document_loaders import TextLoader   

loader = DirectoryLoader(
    path="rag_data",   
    glob="**/*.md",    
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"} 
)

documents = loader.load()

print(f"總共有{len(documents)}筆資料")

總共有13筆資料


## Embedding Model

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5")

In [4]:
# # 匯入 LangChain 的 Embeddings 基底類別
# from langchain_core.embeddings import Embeddings
# # 匯入 OpenAI 客戶端，用於呼叫 LM Studio 的 API
# from openai import OpenAI

# # 自訂 LM Studio 文字嵌入類別，繼承自 LangChain 的 Embeddings
# class LmStudioEmbeddings(Embeddings):
#     def __init__(self, model_name, url):
#         """
#         初始化 LM Studio Embeddings
#         :param model_name: 要使用的嵌入模型名稱
#         :param url: LM Studio 本地或遠端 API 的位址
#         """
#         self.model_name = model_name
#         self.url = url
#         # 建立 OpenAI 客戶端，連接 LM Studio 的 API
#         self.client = OpenAI(base_url=url, api_key="lm-studio")

#     def embed_query(self, text: str):
#         """
#         將單筆文字轉換為向量
#         :param text: 要嵌入的文字
#         :return: 向量 (list of float)
#         """
#         response = self.client.embeddings.create(
#             input=text,      # 傳入單筆文字
#             model=self.model_name  # 指定使用的模型
#         )
#         # 回傳第一筆 embedding
#         return response.data[0].embedding

#     def embed_documents(self, texts: list[str]):
#         """
#         將多筆文字轉換為向量
#         :param texts: 要嵌入的文字列表
#         :return: 向量列表，每個元素對應一筆文字
#         """
#         response = self.client.embeddings.create(
#             input=texts,     # 傳入多筆文字
#             model=self.model_name  # 指定使用的模型
#         )
#         # 回傳每筆文字的 embedding
#         return [x.embedding for x in response.data]

# embeddings = LmStudioEmbeddings(model_name="text-embedding-bge-large-zh-v1.5", url="http://127.0.0.1:1234/v1")

# Multi-Query Retrieval

## 文本分段

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
headers_to_split_on=[
    ("#", "主標題"),
    ("##", "章"),
    ("###", "節"),
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

split_docs = []
for doc in documents:
    docs = splitter.split_text(doc.page_content)
    for d in docs:
        d.metadata.update(doc.metadata)
        split_docs.append(d)


print(f"原始文件數量：{len(documents)}")
print(f"分段後文件數量：{len(split_docs)}")

split_docs[:5]

原始文件數量：13
分段後文件數量：229


[Document(metadata={'主標題': 'Linux 完整簡介', '章': '什麼是 Linux?', 'source': 'rag_data\\Linux簡介.md'}, page_content='Linux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。  \nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。'),
 Document(metadata={'主標題': 'Linux 完整簡介', '章': 'Linux 的歷史背景', 'source': 'rag_data\\Linux簡介.md'}, page_content="要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。  \n1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。  \n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 comp.os.minix 新聞群組發布了一則著名的訊息,宣布他正在開發一個免費的作業系統(只是個人愛好,不會像 GNU 那樣龐大和專業)。這個謙虛的開始最終演變成了今

## 向量資料庫

In [7]:
import uuid
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name=str(uuid.uuid4()),     # collection 名稱（相當於一個資料表）
    embedding_function=embeddings,        # 指定嵌入函式
    # persist_directory="自由指定子資料夾"         # 向量資料儲存路徑
)

vector_store.add_documents(split_docs)
print("✅ 成功新增新資料至 Chroma。")

✅ 成功新增新資料至 Chroma。


In [8]:
vector_store.similarity_search("機器學習", k=3)

[Document(id='aa3b6c37-3c31-4f27-b294-d3eb712bb88d', metadata={'source': 'rag_data\\機器學習的原理.md', '主標題': '機器學習的原理：深度解析'}, page_content='機器學習(Machine Learning)是人工智慧領域中最重要的分支之一,它使計算機系統能夠從數據中學習並改進其性能,而無需被明確編程。本文將深入探討機器學習的核心原理、主要類型、算法機制以及實際應用。'),
 Document(id='491f08a2-b4d0-455b-bb7e-28e31bee9ec8', metadata={'主標題': '機器學習的原理：深度解析', '章': '二、機器學習的主要類型', 'source': 'rag_data\\機器學習的原理.md'}, page_content='機器學習可以根據學習方式的不同分為幾個主要類型,每種類型都有其特定的應用場景和算法。'),
 Document(id='06e7448f-95ee-482c-a7a8-5e2f8ed9de4b', metadata={'主標題': '機器學習的原理：深度解析', '章': '一、機器學習的基本概念', 'source': 'rag_data\\機器學習的原理.md'}, page_content='機器學習的核心思想是讓計算機通過經驗自動改進。與傳統編程不同,傳統編程需要人類明確地告訴計算機每一步該做什麼,而機器學習則是讓計算機從大量數據中找出規律,並根據這些規律做出決策或預測。')]

## 建立 Multi-Query RAG 系統

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = CommaSeparatedListOutputParser()
format_instructions = parser.get_format_instructions()

system_prompt = f'''
你是一個搜尋專家。請針對下方的原始提問，生成{{n}}個不同維度的搜尋查詢句（Search Queries），以優化 RAG 系統的檢索結果。

{format_instructions}
'''

user_prompt = '''
**原始提問：** {question}
'''

multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

multi_query_prompt.invoke({"question": "什麼是 RAG？", "n": 3})


ChatPromptValue(messages=[SystemMessage(content='\n你是一個搜尋專家。請針對下方的原始提問，生成3個不同維度的搜尋查詢句（Search Queries），以優化 RAG 系統的檢索結果。\n\nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n**原始提問：** 什麼是 RAG？\n', additional_kwargs={}, response_metadata={})])

In [ ]:
multi_query_chain = multi_query_prompt | llm | parser

In [11]:
multi_query_chain.invoke({"question": "機器學習的好處", "n": 3})

['機器學習優勢', '機器學習應用場景', '機器學習收益分析']

In [12]:
def multi_query_retrieval(question, n=3):
    querys = multi_query_chain.invoke({"question": question, "n": n})
    documents = []
    seen_ids = set()
    for query in querys:
        docs = vector_store.similarity_search(query, k=5)
        for doc in docs:
            if doc.id not in seen_ids:
                seen_ids.add(doc.id)
                documents.append(doc)
    
    return documents

multi_query_retrieval("機器學習的好處", n=3)

[Document(id='aa3b6c37-3c31-4f27-b294-d3eb712bb88d', metadata={'source': 'rag_data\\機器學習的原理.md', '主標題': '機器學習的原理：深度解析'}, page_content='機器學習(Machine Learning)是人工智慧領域中最重要的分支之一,它使計算機系統能夠從數據中學習並改進其性能,而無需被明確編程。本文將深入探討機器學習的核心原理、主要類型、算法機制以及實際應用。'),
 Document(id='0f53c092-f715-40f3-b1d8-9c2c6b9f2001', metadata={'主標題': '機器學習的原理：深度解析', 'source': 'rag_data\\機器學習的原理.md', '章': '結語'}, page_content='機器學習是一個快速發展的領域,新的算法和技術不斷湧現。理解機器學習的基本原理對於在這個領域深入發展至關重要。從數據的準備,到模型的選擇和訓練,再到結果的評估和優化,每一步都需要仔細考慮。隨著計算能力的提升和數據的積累,機器學習將在更多領域展現其強大的潛力,為人類社會帶來更多的創新和變革。'),
 Document(id='c55c9498-3c32-455a-aaf5-1c2332073c96', metadata={'主標題': '機器學習的原理：深度解析', '章': '七、機器學習的實際應用', 'source': 'rag_data\\機器學習的原理.md'}, page_content='機器學習已經滲透到我們生活的方方面面,改變了許多行業的運作方式。  \n在醫療健康領域,機器學習被用於疾病診斷、藥物研發、個性化治療等。例如,深度學習模型可以分析醫學影像,幫助醫生更準確地診斷癌症等疾病。  \n在金融領域,機器學習被用於信用評分、欺詐檢測、算法交易等。銀行使用機器學習模型來評估貸款申請者的信用風險,保險公司用它來定價保單。  \n在電子商務領域,推薦系統是機器學習最成功的應用之一。亞馬遜、Netflix等公司使用複雜的推薦算法,根據用戶的歷史行為和偏好推薦商品或內容,這不僅提升了用戶體驗,也顯著增加了銷售額。  \n在自動駕駛領域,機器學習,特別是深度學習,在感知、決策和控制等環節都發揮著關鍵

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
你是一位耐心且專業的 AI 助教，負責回答學生的問題。

請根據「提供的參考資料（context）」來回答問題：
- 只能使用 context 中的資訊作答
- 不要自行補充未出現在 context 中的知識
- 回答時請使用清楚、白話、教學導向的說明方式
- 不要讓學生知道參考資料
- 問答跟參考資料無關時拒絕回答 

參考資料：
{context}
"""

user_prompt = """
問題：
{question}
"""

question_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

# 範例
question_prompt.invoke({
    "question": "什麼是 RAG？",
    "context": "RAG 是一種結合檢索（Retrieval）與生成（Generation）的人工智慧架構，用來提升回答的準確性。"
})


ChatPromptValue(messages=[SystemMessage(content='\n你是一位耐心且專業的 AI 助教，負責回答學生的問題。\n\n請根據「提供的參考資料（context）」來回答問題：\n- 只能使用 context 中的資訊作答\n- 不要自行補充未出現在 context 中的知識\n- 回答時請使用清楚、白話、教學導向的說明方式\n- 不要讓學生知道參考資料\n- 問答跟參考資料無關時拒絕回答 \n\n參考資料：\nRAG 是一種結合檢索（Retrieval）與生成（Generation）的人工智慧架構，用來提升回答的準確性。\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n問題：\n什麼是 RAG？\n', additional_kwargs={}, response_metadata={})])

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda 

multi_query_parallel = RunnableParallel(
    question=RunnablePassthrough(),
    context=RunnableLambda(
        lambda question: multi_query_retrieval(question=question, n=3)
    )
)

multi_query_parallel.invoke("機器學習的好處")


{'question': {'question': '機器學習的好處'},
 'context': [Document(id='aa3b6c37-3c31-4f27-b294-d3eb712bb88d', metadata={'source': 'rag_data\\機器學習的原理.md', '主標題': '機器學習的原理：深度解析'}, page_content='機器學習(Machine Learning)是人工智慧領域中最重要的分支之一,它使計算機系統能夠從數據中學習並改進其性能,而無需被明確編程。本文將深入探討機器學習的核心原理、主要類型、算法機制以及實際應用。'),
  Document(id='0f53c092-f715-40f3-b1d8-9c2c6b9f2001', metadata={'主標題': '機器學習的原理：深度解析', '章': '結語', 'source': 'rag_data\\機器學習的原理.md'}, page_content='機器學習是一個快速發展的領域,新的算法和技術不斷湧現。理解機器學習的基本原理對於在這個領域深入發展至關重要。從數據的準備,到模型的選擇和訓練,再到結果的評估和優化,每一步都需要仔細考慮。隨著計算能力的提升和數據的積累,機器學習將在更多領域展現其強大的潛力,為人類社會帶來更多的創新和變革。'),
  Document(id='c55c9498-3c32-455a-aaf5-1c2332073c96', metadata={'source': 'rag_data\\機器學習的原理.md', '章': '七、機器學習的實際應用', '主標題': '機器學習的原理：深度解析'}, page_content='機器學習已經滲透到我們生活的方方面面,改變了許多行業的運作方式。  \n在醫療健康領域,機器學習被用於疾病診斷、藥物研發、個性化治療等。例如,深度學習模型可以分析醫學影像,幫助醫生更準確地診斷癌症等疾病。  \n在金融領域,機器學習被用於信用評分、欺詐檢測、算法交易等。銀行使用機器學習模型來評估貸款申請者的信用風險,保險公司用它來定價保單。  \n在電子商務領域,推薦系統是機器學習最成功的應用之一。亞馬遜、Netflix等公司使用複雜的推薦算法,根據用戶的歷史行為和偏好推薦商品或內容,這不僅提升了用戶體驗,也

In [15]:
multi_query_rag_chain = multi_query_parallel | question_prompt | llm | StrOutputParser()

In [16]:
answer = multi_query_rag_chain.invoke('機器學習的好處')
print(answer)

機器學習的優點在於它使計算機系統能夠從數據中學習並改進其性能，無需明確編程。 此外，機器學習已經滲透到我們生活的方方面面，改變了許多行業的運作方式，例如在醫療健康領域用於疾病診斷、在金融領域用於信用評分等。


In [17]:
answer = multi_query_rag_chain.invoke('LINUX是免費的嗎')
print(answer)

是的，Linux 是免費的。根據提供的資料，Linux 是一個自由且開源的類 Unix 作業系統核心(Kernel)，並採用 GNU General Public License (GPL) 授權，這確保了它永遠保持自由，任何人都可以查看、修改和分發 Linux 的原始碼。


In [18]:
answer = multi_query_rag_chain.invoke('台北天氣')
print(answer)

抱歉，我無法回答關於台北天氣的問題。提供的參考資料只包含關於全球氣候變化的資訊，沒有任何關於特定地區天氣的資訊。


In [19]:
answer = multi_query_rag_chain.invoke('台北市旅遊景點')
print(answer)

抱歉，提供的參考資料中沒有關於台北市旅遊景點的資訊。我只能回答與這些文件內容相關的問題。


# Parent-Document Retrieval

# 文件分段

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)


total_docs = []

# 分割父層文件
parent_docs = parent_splitter.split_documents(documents)

for doc in parent_docs:
    doc.metadata['id'] = str(uuid.uuid4())
    doc.metadata['parent_id'] = doc.metadata['id']
    # 分割子層文件
    split_docs = child_splitter.split_documents([doc])
    for sdoc in split_docs:
        sdoc.metadata['id'] = str(uuid.uuid4())
        sdoc.metadata['parent_id'] = doc.metadata['id']
    total_docs.append(doc)
    total_docs.extend(split_docs)
        

print(f"原始文件數量：{len(documents)}")
print(f"父層文件數量：{len(parent_docs)}")
print(f"分段後文件數量：{len(total_docs)}")

total_docs[:5]

原始文件數量：13
父層文件數量：55
分段後文件數量：559


[Document(metadata={'source': 'rag_data\\Linux簡介.md', 'id': 'dc066457-a595-4d2d-9b25-f7fd291cfb50', 'parent_id': 'dc066457-a595-4d2d-9b25-f7fd291cfb50'}, page_content="# Linux 完整簡介\n\n## 什麼是 Linux?\n\nLinux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。\n\nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。\n\n## Linux 的歷史背景\n\n要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。\n\n1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。\n\n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 comp.os.minix 新聞群組發布了一則著名的訊息,宣布他正在開發一個免費的作業系統(只是個人愛好,不會像 GNU 那樣龐大和專業)。這個謙虛的開始最終演變成了今

## 向量資料庫

In [33]:
import uuid
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name=str(uuid.uuid4()),     # collection 名稱（相當於一個資料表）
    embedding_function=embeddings,        # 指定嵌入函式
    # persist_directory="自由指定子資料夾"         # 向量資料儲存路徑
)

vector_store.add_documents(total_docs)
print("✅ 成功新增新資料至 Chroma。")

✅ 成功新增新資料至 Chroma。


## 建立 Parent-Document RAG 系統

In [16]:
def parent_document_retrieval(question, n=20, k=3):
    docs = vector_store.similarity_search(question, k=n)
    seen_ids = set()
    documents = []
    for doc in docs:
        if doc.metadata["parent_id"] not in seen_ids:
            seen_ids.add(doc.metadata["parent_id"])
            parent_docs = vector_store.similarity_search(query="",k=1,filter={"id": doc.metadata["parent_id"]})
            if len(parent_docs) > 0:
                documents.append(parent_docs[0])

    return documents[:k]

parent_document_retrieval("什麼是 Linux?")


[Document(id='7f65deca-c651-4c97-b1d2-507277c0a3d7', metadata={'parent_id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087', 'source': 'rag_data\\Linux簡介.md', 'id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087'}, page_content="# Linux 完整簡介\n\n## 什麼是 Linux?\n\nLinux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。\n\nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。\n\n## Linux 的歷史背景\n\n要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。\n\n1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。\n\n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 comp.os.minix 新聞群組發布了一則著名的訊息,宣布他正在開發一個免費的

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
你是一位耐心且專業的 AI 助教，負責回答學生的問題。

請根據「提供的參考資料（context）」來回答問題：
- 只能使用 context 中的資訊作答
- 不要自行補充未出現在 context 中的知識
- 回答時請使用清楚、白話、教學導向的說明方式
- 不要讓學生知道參考資料
- 問答跟參考資料無關時拒絕回答 

參考資料：
{context}
"""

user_prompt = """
問題：
{question}
"""

question_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

# 範例
question_prompt.invoke({
    "question": "什麼是 RAG？",
    "context": "RAG 是一種結合檢索（Retrieval）與生成（Generation）的人工智慧架構，用來提升回答的準確性。"
})


ChatPromptValue(messages=[SystemMessage(content='\n你是一位耐心且專業的 AI 助教，負責回答學生的問題。\n\n請根據「提供的參考資料（context）」來回答問題：\n- 只能使用 context 中的資訊作答\n- 不要自行補充未出現在 context 中的知識\n- 回答時請使用清楚、白話、教學導向的說明方式\n- 不要讓學生知道參考資料\n- 問答跟參考資料無關時拒絕回答 \n\n參考資料：\nRAG 是一種結合檢索（Retrieval）與生成（Generation）的人工智慧架構，用來提升回答的準確性。\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n問題：\n什麼是 RAG？\n', additional_kwargs={}, response_metadata={})])

In [18]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda 

parent_document_parallel = RunnableParallel(
    question=RunnablePassthrough(),
    context=RunnableLambda(
        lambda x: parent_document_retrieval(question=x,n=20)
    )
)

parent_document_parallel.invoke("Linux是免費的嗎？")


{'question': 'Linux是免費的嗎？',
 'context': [Document(id='7f65deca-c651-4c97-b1d2-507277c0a3d7', metadata={'parent_id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087', 'id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087', 'source': 'rag_data\\Linux簡介.md'}, page_content="# Linux 完整簡介\n\n## 什麼是 Linux?\n\nLinux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。\n\nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。\n\n## Linux 的歷史背景\n\n要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。\n\n1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。\n\n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 c

In [19]:
parent_document_rag_chain = parent_document_parallel | question_prompt | llm | StrOutputParser()

In [46]:
answer = parent_document_rag_chain.invoke('機器學習的好處')
print(answer)

根據提供的參考資料，機器學習的主要好處包括：

*   **從數據中學習並改進性能：** 機器學習的核心思想是讓計算機系統通過分析數據自動改進其在特定任務上的性能，無需被明確編程。
*   **發現隱藏的結構或模式：** 非監督式學習可以從沒有標籤的數據中發現隱藏的結構或模式，例如聚類分析可以將相似的數據點分組在一起。
*   **提升模型性能：** 好的特徵工程可以顯著提升模型的性能。


In [47]:
answer = parent_document_rag_chain.invoke('LINUX是免費的嗎')
print(answer)

是的，Linux 是免費的。

根據提供的資料，Linux 的開源性質意味著「沒有授權費用」，這使得企業和個人可以節省大量成本。同時，GPL 授權確保了 Linux 永遠保持自由，任何基於 Linux 的衍生作品也必須保持開源。


In [48]:
answer = parent_document_rag_chain.invoke('台北天氣')
print(answer)

抱歉，我無法回答關於台北天氣的問題。參考資料中只包含全球氣候變化的資訊，沒有任何關於天氣或特定城市天氣的內容。


# Multi-Vector Retrieval

In [6]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

class MultiVectorEmbedding(BaseModel):
    text: str = Field(description="不同語意視角的內容")
    type: Literal[
        "concept_vector",
        "qa_vector",
        "process_vector",
        "application_vector",
        "keyword_vector",
    ] = Field(description="向量的類型")


parser = PydanticOutputParser(pydantic_object=MultiVectorEmbedding)
format_instructions = parser.get_format_instructions()

system_prompt = """
你是一個專門為「向量檢索系統（Vector Retrieval / RAG）」準備資料的助理。

請根據下面提供的【父層文件內容】，從「不同語意視角」切分並改寫出多組文字片段，
每一組文字都將被用來產生一個 embedding 向量。

【父層文件內容】
---
{parent_document}
---

請依照以下規則產生多組切片（每一組都是獨立的一段文字）：

1. 概念定義向量(concept_vector)
   - 將文件中「核心概念、名詞定義、關鍵說明」整理成一段簡潔文字  
   - 適合用於回答「什麼是 X？」類型問題

2. 問題導向向量(qa_vector)  
   - 將內容改寫成「可能被使用者詢問的問題＋對應重點答案」  
   - 適合用於自然語言查詢檢索

3. 步驟／流程向量(process_vector)
   - 若文件包含流程、方法、步驟，請將其整理成清楚的操作或邏輯流程描述  
   - 適合用於「怎麼做」「流程是什麼」類型查詢

4. 應用與情境向量(application_vector)  
   - 將內容轉換為實際應用場景、使用案例或情境說明  
   - 適合用於「可以用在什麼地方」的查詢

5. 關鍵詞／摘要向量(keyword_vector)  
   - 使用偏向關鍵字密集、摘要式的寫法  
   - 適合提升語意召回率（recall）

注意事項：
- 所有內容必須來自父層文件
- 不可引入外部知識
- 不要加入任何說明文字
- 每一欄內容都應可獨立產生 embedding

依照以下規則產生多組切片
請嚴格遵守以下輸出格式：
{format_instructions}
"""

multi_vector_prompt = ChatPromptTemplate(
    [
        ("user", system_prompt),
    ],
    partial_variables={
        "format_instructions": format_instructions
    }
)


multi_vector_prompt.invoke(documents[0].page_content)


ChatPromptValue(messages=[HumanMessage(content='\n你是一個專門為「向量檢索系統（Vector Retrieval / RAG）」準備資料的助理。\n\n請根據下面提供的【父層文件內容】，從「不同語意視角」切分並改寫出多組文字片段，\n每一組文字都將被用來產生一個 embedding 向量。\n\n【父層文件內容】\n---\n# Linux 完整簡介\n\n## 什麼是 Linux?\n\nLinux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。\n\nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。\n\n## Linux 的歷史背景\n\n要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。\n\n1983 年,Richard Stallman 啟動了 GNU 專案(GNU\'s Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。\n\n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 comp.os.minix 新聞群組發布了一則著名的訊息,宣布他正在開發一個免費的作業系統(只是個人愛好,不會像 GNU 那

In [7]:
multi_vector_chain = multi_vector_prompt | llm | JsonOutputParser()

In [ ]:
multi_vector_chain.invoke(documents[0].page_content)


[{'text': 'Linux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。',
  'type': 'concept_vector'},
 {'text': 'Linux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。',
  'type': 'concept_vector'},
 {'text': '要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。',
  'type': 'concept_vector'},
 {'text': "1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。",
  'type': 'concept_vector'},
 {'text': 'Linux 最重要的特性就是它的開源性質。任何人都可以自由查看、修改和分發 Linux 的原始碼。',
  'type': 'concept_vector'},
 {'text': 'Linux 從設計之初就支援多使用者同時使用系統。每個使用者都有自己的帳號、家目錄和權限設定,系統管理員(root)可以控制誰能存取什麼資源。',
  'type': 'concept_vector'},
 {'text': 'Linux 的穩定性以其卓越的穩定性聞名,伺服器可以連續運行數年而不需要重新啟動(除了核心更新)。',
  'type': 'concept_vector'},
 {'text': 'Debian 家族是最古老且最有影響力的發行版家族之一。Debian 以其穩定性和自由軟體承諾著稱,採用 APT 套件管理系統。',
  'type': 'concept_vector'},
 {'text': 'Red Hat 家族主要面向企業市場。Red Hat Ente

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid
from tqdm import tqdm

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=200)

total_docs = []

# 分割父層文件
parent_docs = parent_splitter.split_documents(documents)

for doc in tqdm(parent_docs, desc="處理父層文件"):
    doc.metadata['id'] = str(uuid.uuid4())
    doc.metadata['parent_id'] = doc.metadata['id']
    total_docs.append(doc)
    sub_embedding_docs = multi_vector_chain.invoke(doc.page_content)
    for sdoc in sub_embedding_docs:
        d = Document(page_content=sdoc['text'], metadata=doc.metadata)
        d.metadata['id'] = str(uuid.uuid4())
        d.metadata['parent_id'] = doc.metadata['id']
        d.metadata['type'] = sdoc['type']
        total_docs.append(d)
        

print(f"原始文件數量：{len(documents)}")
print(f"父層文件數量：{len(parent_docs)}")
print(f"子向量文件數量：{len(total_docs)}")

total_docs[:5]

處理父層文件: 100%|██████████| 55/55 [43:26<00:00, 47.40s/it]

原始文件數量：13
父層文件數量：55
子向量文件數量：799


[Document(metadata={'source': 'rag_data\\Linux簡介.md', 'id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087', 'parent_id': 'fc5ea6cf-e3c4-42a7-9e51-3fc409e68087'}, page_content="# Linux 完整簡介\n\n## 什麼是 Linux?\n\nLinux 是一個自由且開源的類 Unix 作業系統核心(Kernel),由芬蘭電腦科學家 Linus Torvalds 在 1991 年首次發布。它不僅僅是一個作業系統核心,更代表了一種軟體開發哲學和全球協作的典範。今天,Linux 已經成為世界上最重要的作業系統之一,從超級電腦、伺服器、個人電腦到智慧型手機和嵌入式設備,都能看到它的身影。\n\nLinux 的名稱結合了創始人 Linus 的名字和 Unix 作業系統的名稱。雖然 Linux 本身只是作業系統的核心,但通常人們會將完整的作業系統(包含核心、系統工具、應用程式等)統稱為 Linux 或 GNU/Linux。這是因為大多數 Linux 發行版都使用了 GNU 專案提供的大量系統工具和軟體。\n\n## Linux 的歷史背景\n\n要理解 Linux 的誕生,我們需要回顧一些電腦歷史。在 1960 和 1970 年代,Unix 作業系統在貝爾實驗室誕生,並逐漸在學術界和商業領域流行。Unix 以其簡潔的設計哲學和強大的功能著稱,但隨著時間推移,Unix 變成了專有軟體,需要昂貴的授權費用。\n\n1983 年,Richard Stallman 啟動了 GNU 專案(GNU's Not Unix 的遞迴縮寫),目標是創建一個完全自由的類 Unix 作業系統。GNU 專案開發了許多重要的工具,如 GCC 編譯器、Bash shell、Emacs 編輯器等,但一直缺少一個可用的作業系統核心。\n\n1991 年,當時還是赫爾辛基大學學生的 Linus Torvalds 開始開發自己的作業系統核心,最初只是作為個人專案。他在同年 8 月 25 日在 comp.os.minix 新聞群組發布了一則著名的訊息,宣布他正在開發一個免費的作業系統(只是個人愛好,不會像 GNU 那樣龐大和專業)。這個謙虛的開始最終演變成了今

In [14]:
from langchain_chroma import Chroma
# 建立或載入現有的 Chroma 向量資料庫
vector_store = Chroma(
    collection_name="multivector",     # collection 名稱（相當於一個資料表）
    embedding_function=embeddings,        # 指定嵌入函式
    persist_directory="./rag_db"         # 向量資料儲存路徑
)

In [ ]:
# # 清空現有 collection
# vector_store.delete_collection()

# # 重新建立一個新的空 collection
# vector_store = Chroma(
#     collection_name="multivector",     # collection 名稱（相當於一個資料表）
#     embedding_function=embeddings,        # 指定嵌入函式
#     persist_directory="./rag_db"         # 向量資料儲存路徑
# )

In [15]:
# 新增文件資料
vector_store.add_documents(total_docs)

print("✅ 成功新增新資料至 Chroma。")

✅ 成功新增新資料至 Chroma。


In [22]:
# 以下的程式基本上跟 parent_document_retrieval 一樣，只有修改函數名稱
def multi_vector_retrieval(question, n=20, k=3):
    docs = vector_store.similarity_search(question, k=n)
    seen_ids = set()
    documents = []
    for doc in docs:
        if doc.metadata["parent_id"] not in seen_ids:
            seen_ids.add(doc.metadata["parent_id"])
            parent_docs = vector_store.similarity_search(query="",k=1,filter={"id": doc.metadata["parent_id"]})
            if len(parent_docs) > 0:
                documents.append(parent_docs[0])

    return documents[:k]


from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
你是一位耐心且專業的 AI 助教，負責回答學生的問題。

請根據「提供的參考資料（context）」來回答問題：
- 只能使用 context 中的資訊作答
- 不要自行補充未出現在 context 中的知識
- 回答時請使用清楚、白話、教學導向的說明方式
- 不要讓學生知道參考資料
- 問答跟參考資料無關時拒絕回答 

參考資料：
{context}
"""

user_prompt = """
問題：
{question}
"""

question_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

# 範例
question_prompt.invoke({
    "question": "什麼是 RAG？",
    "context": "RAG 是一種結合檢索（Retrieval）與生成（Generation）的人工智慧架構，用來提升回答的準確性。"
})


from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda 

multi_vector_parallel = RunnableParallel(
    question=RunnablePassthrough(),
    context=RunnableLambda(
        lambda x: multi_vector_retrieval(question=x,n=20)
    )
)

multi_vector_rag_chain = multi_vector_parallel | question_prompt | llm | StrOutputParser()


In [23]:
answer = multi_vector_rag_chain.invoke('機器學習的好處')
print(answer)

機器學習使計算機系統能夠從數據中學習並改進其性能，而無需被明確編程。這意味著它能自動找出數據中的規律，並根據這些規律做出決策或預測，相較於傳統的編程方式，可以讓計算機更靈活地應對複雜的問題。

機器學習已經滲透到我們生活的方方面面，改變了許多行業的運作方式，例如在醫療健康、金融和電子商務等領域都有廣泛應用，為社會帶來創新和變革。


In [24]:
answer = multi_vector_rag_chain.invoke('LINUX是免費的嗎')
print(answer)

是的，Linux 是免費的。它是一個自由且開源的作業系統核心，任何人都可以自由查看、修改和分發它的原始碼。這意味著你可以免費使用 Linux，而且不會有授權費用。


In [25]:
answer = multi_vector_rag_chain.invoke('台北天氣')
print(answer)

很抱歉，我只能根據提供的資料回答問題。目前我的知識庫中沒有關於台北天氣的信息。


In [26]:
answer = multi_vector_rag_chain.invoke('經濟成長的來源？')
print(answer)

根據理論，長期經濟成長是提升生活水準的根本途徑。經濟成長的主要來源有以下幾個因素：

*   **資本累積**：增加生產所需的工具和設備。
*   **勞動力增加**：更多的人參與到生產中來。
*   **人力資本提升**：透過教育、訓練等方式提高勞動力的知識和技能。
*   **技術進步**：創造新的產品、服務或更有效率的生產方法。

傳統理論強調資本累積的重要性，但同時也指出，僅靠資本累積無法維持長期成長，技術進步才是長期經濟成長的關鍵。


In [27]:
answer = multi_vector_rag_chain.invoke('動漫產業面臨的最大挑戰有哪些？')
print(answer)

動漫產業目前面臨著幾個主要的挑戰：

1. **製作成本上升與從業者待遇不匹配：** 動漫製作公司的成本在增加，但從業者的薪資卻跟不上，導致人才流失。
2. **版權保護和盜版問題：** 盜版網站和非法下載對產業造成了很大的經濟損失。
3. **內容同質化和創意枯竭：** 作品的題材、設定和劇情容易變得相似，讓觀眾產生審美疲勞。
4. **國際競爭加劇：** 中國、韓國等國家動漫產業的快速發展，以及歐美科技巨頭的加入，都對日本動漫產業帶來了壓力。
